## RAG Pipeline: Data Ingestion to Vector DB Pipeline

In [9]:
import os
from langchain_community.document_loaders import PyPDFLoader,PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pathlib import Path

In [14]:
def process_all_pdfs(pdf_directory):
    """Process all PDF files in a directory"""
    all_documents = []
    pdf_dir = Path(pdf_directory)

    pdf_files = list(pdf_dir.glob("**/*.pdf"))

    print(f"Found {len(pdf_files)} PDF files to process")

    for pdf_file in pdf_files:
        print(f"\nProcessing: {pdf_file.name}")
        try:
            loader = PyPDFLoader(str(pdf_file))
            documents = loader.load()

            # Add source information to metadata
            for doc in documents:
                doc.metadata['source_file']=pdf_file.name
                doc.metadata['file_type']='pdf'

            all_documents.extend(documents)
            print(f"✅ Loaded {len(documents)} pages")
        except Exception as e:
            print(f"❌ Error: {e}")

    print(f"\nTotal documents loaded: {len(documents)}")
    return all_documents



In [15]:
# Process all PDFs in the data directory
all_pdf_documents = process_all_pdfs("../data")
all_pdf_documents

Found 3 PDF files to process

Processing: AI_ML_DL.pdf
✅ Loaded 5 pages

Processing: Large_Language_Models.pdf
✅ Loaded 6 pages

Processing: RAG_MCP.pdf
✅ Loaded 7 pages

Total documents loaded: 7


[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-10-31T17:45:45+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-10-31T17:45:45+00:00', 'subject': '(unspecified)', 'title': 'AI, Machine Learning & Deep Learning', 'trapped': '/False', 'source': '..\\data\\pdf\\AI_ML_DL.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'source_file': 'AI_ML_DL.pdf', 'file_type': 'pdf'}, page_content='Study notes \x7f concise revision edition\nPage 1\n AI, Machine Learning & Deep Learning\n 5 pages \x7f foundations, learning paradigms, algorithms, neural networks and CNNs\n1. AI, ML & DL Foundations\nArtificial Intelligence (AI)\nAI is the broad field of building systems that perform tasks associated with human intelligence, such as reasoning, perception,\nlanguage understanding and decision-making.\nMachine Learning (ML)\nML is a subset of AI in which systems learn patterns from data to make predictions or decis

In [16]:
### Text Splitting get into chunks

def split_documents(documents,chunk_size=1000,chunk_overlap=200):
    """Split documents into smaller chunks for better RAG performance"""
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        length_function=len,
        separators=["\n\n","\n"," ",""]
    )
    split_docs = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} documents into {len(split_docs)} chunks")

    # Show example of a chunk
    if split_docs:
        print(f"\nExample chunk")
        print(f"Content: {split_docs[0].page_content[:200]}...")
        print(f"Metadata: {split_docs[0].metadata}")

    return split_docs

In [17]:
chunks = split_documents(all_pdf_documents)
chunks

Split 18 documents into 36 chunks

Example chunk
Content: Study notes  concise revision edition
Page 1
 AI, Machine Learning & Deep Learning
 5 pages  foundations, learning paradigms, algorithms, neural networks and CNNs
1. AI, ML & DL Foundations
Artifici...
Metadata: {'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-10-31T17:45:45+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-10-31T17:45:45+00:00', 'subject': '(unspecified)', 'title': 'AI, Machine Learning & Deep Learning', 'trapped': '/False', 'source': '..\\data\\pdf\\AI_ML_DL.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'source_file': 'AI_ML_DL.pdf', 'file_type': 'pdf'}


[Document(metadata={'producer': 'ReportLab PDF Library - (opensource)', 'creator': '(unspecified)', 'creationdate': '2026-10-31T17:45:45+00:00', 'author': '(anonymous)', 'keywords': '', 'moddate': '2026-10-31T17:45:45+00:00', 'subject': '(unspecified)', 'title': 'AI, Machine Learning & Deep Learning', 'trapped': '/False', 'source': '..\\data\\pdf\\AI_ML_DL.pdf', 'total_pages': 5, 'page': 0, 'page_label': '1', 'source_file': 'AI_ML_DL.pdf', 'file_type': 'pdf'}, page_content='Study notes \x7f concise revision edition\nPage 1\n AI, Machine Learning & Deep Learning\n 5 pages \x7f foundations, learning paradigms, algorithms, neural networks and CNNs\n1. AI, ML & DL Foundations\nArtificial Intelligence (AI)\nAI is the broad field of building systems that perform tasks associated with human intelligence, such as reasoning, perception,\nlanguage understanding and decision-making.\nMachine Learning (ML)\nML is a subset of AI in which systems learn patterns from data to make predictions or decis